# 06.10 · Recommender systems: popularity and collaborative filtering

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/06-unsupervised/10-recommenders/lesson.ipynb)

Read the [README](README.md) first and do the 4-customer cosine calculation on paper. This notebook checks it, builds "customers who bought X also bought" for ShopWave, and measures whether it beats simply recommending best-sellers.

In [1]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")

## 1. Four customers, five products

In [2]:
M = pd.DataFrame(
    [[1, 1, 0, 1, 0],
     [1, 1, 0, 0, 0],
     [0, 1, 1, 0, 1],
     [1, 0, 0, 1, 1]],
    index=["Ana", "Ben", "Chi", "Dev"],
    columns=["mug", "teapot", "candle", "coaster", "lantern"])
M

,mug,teapot,candle,coaster,lantern
Ana,1,1,0,1,0
Ben,1,1,0,0,0
Chi,0,1,1,0,1
Dev,1,0,0,1,1


## 2. Cosine similarity between products, by hand and in code

In [3]:
def cosine(a, b):
    return (a @ b) / (np.linalg.norm(a) * np.linalg.norm(b))

for p, q in [("mug", "coaster"), ("teapot", "candle"), ("mug", "candle")]:
    a, b = M[p].values, M[q].values
    print(f"{p:>7} vs {q:<8} dot {a @ b}, |a| {np.linalg.norm(a):.3f}, |b| {np.linalg.norm(b):.3f}"
          f"  -> cosine {cosine(a, b):.3f}")

    mug vs coaster  dot 2, |a| 1.732, |b| 1.414  -> cosine 0.816
 teapot vs candle   dot 1, |a| 1.732, |b| 1.000  -> cosine 0.577
    mug vs candle   dot 0, |a| 1.732, |b| 1.000  -> cosine 0.000


In [4]:
from sklearn.metrics.pairwise import cosine_similarity
item_sim = pd.DataFrame(cosine_similarity(M.T), index=M.columns, columns=M.columns)
item_sim.round(3)

,mug,teapot,candle,coaster,lantern
mug,1.000,0.667,0.000,0.816,0.408
teapot,0.667,1.000,0.577,0.408,0.408
candle,0.000,0.577,1.000,0.000,0.707
coaster,0.816,0.408,0.000,1.000,0.500
lantern,0.408,0.408,0.707,0.500,1.000


## 3. Recommend for Ben

Ben bought a mug and a teapot. Score every product he has *not* bought by adding up its similarity to each product he has.

In [5]:
owned = M.loc["Ben"] == 1
scores = item_sim.loc[owned[owned].index].sum()      # sum over mug and teapot rows
print(scores[~owned].sort_values(ascending=False).round(3))

coaster    1.225
lantern    0.816
candle     0.577
dtype: float64


Coaster wins (0.816 + 0.408 ≈ 1.22): Ana and Dev, who bought mugs, also bought coasters. That is **item-item collaborative filtering**: no product descriptions, only who bought what.

## 4. ShopWave: who bought what

A fair test needs a fair split. For every customer, the **last** invoice is hidden; the recommender learns only from earlier invoices. Then we ask: does the top-10 list contain anything the customer actually bought next time?

We keep the 500 best-selling products to keep the matrix small.

In [6]:
# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")

import sys
sys.path.insert(0, str(data_path("download.py").parent))
from download import load_retail

raw = load_retail("Year 2010-2011")
sales = raw[~raw["Invoice"].astype(str).str.startswith("C")]
sales = sales[(sales["Quantity"] > 0) & (sales["Price"] > 0)].dropna(subset=["Customer ID"]).drop_duplicates()
sales = sales.assign(Revenue=sales["Quantity"] * sales["Price"], **{"Customer ID": sales["Customer ID"].astype(int)})
print(f"{len(sales):,} clean transaction lines, {sales['Customer ID'].nunique():,} customers")

392,693 clean transaction lines, 4,338 customers


In [7]:
# Hold out each customer's LAST invoice; learn from everything before it.
last_invoice = sales.sort_values("InvoiceDate").groupby("Customer ID")["Invoice"].last()
is_last = sales["Invoice"].isin(set(last_invoice))
train, test = sales[~is_last], sales[is_last]

top_products = train["StockCode"].value_counts().head(500).index
tr = train[train["StockCode"].isin(top_products)]
R = pd.crosstab(tr["Customer ID"], tr["StockCode"]).clip(upper=1)     # 1 = bought at least once
names = sales.drop_duplicates("StockCode").set_index("StockCode")["Description"]
print(f"train: {R.shape[0]:,} customers x {R.shape[1]} products, {R.values.mean():.1%} of cells are 1")

train: 2,805 customers x 500 products, 6.8% of cells are 1


In [8]:
from sklearn.metrics.pairwise import cosine_similarity
S = cosine_similarity(R.T.values)          # product x product
np.fill_diagonal(S, 0)                     # a product is not its own recommendation
sim = pd.DataFrame(S, index=R.columns, columns=R.columns)

for code_ in ["85123A", "22423", "84879"]:
    print(f"\n{names[code_]}:")
    for other, s in sim[code_].nlargest(5).items():
        print(f"   {s:.2f}  {names[other]}")


WHITE HANGING HEART T-LIGHT HOLDER:
   0.53  RED HANGING HEART T-LIGHT HOLDER
   0.51  CANDLEHOLDER PINK HANGING HEART
   0.43  HEART OF WICKER LARGE
   0.41  HEART OF WICKER SMALL
   0.41  WOODEN PICTURE FRAME WHITE FINISH

REGENCY CAKESTAND 3 TIER:
   0.54  ROSES REGENCY TEACUP AND SAUCER 
   0.52  GREEN REGENCY TEACUP AND SAUCER
   0.50  PINK REGENCY TEACUP AND SAUCER
   0.47  REGENCY TEAPOT ROSES 
   0.45  SET OF 3 REGENCY CAKE TINS

ASSORTED COLOUR BIRD ORNAMENT:
   0.47  PAINTED METAL PEARS ASSORTED
   0.36  REGENCY CAKESTAND 3 TIER
   0.35  WHITE HANGING HEART T-LIGHT HOLDER
   0.32  SPOTTY BUNTING
   0.32  HOME BUILDING BLOCK WORD


These read like a sensible shop assistant: hanging heart T-light holders go with other hanging-heart products, the Regency cake stand with Regency teacups and teapots. Nobody told the algorithm about colours or ranges; it learned them from baskets.

## 5. Does it beat best-sellers?

**Popularity baseline**: recommend the 10 best-selling products the customer has not bought yet. Always build this first; it is surprisingly hard to beat.

**Hit rate @10**: the share of customers for whom at least one of the 10 recommendations appears in their held-out last invoice.

In [9]:
truth = test[test["StockCode"].isin(top_products)].groupby("Customer ID")["StockCode"].apply(set)
customers = [c for c in R.index if c in truth.index]
products = np.array(R.columns)
bought = R.values > 0
popularity = R.sum().values.astype(float)

def hit_rate(score_matrix, k=10):
    hits = 0
    for c in customers:
        i = R.index.get_loc(c)
        s = score_matrix[i].copy(); s[bought[i]] = -np.inf          # never recommend what they own
        top_k = products[np.argsort(-s)[:k]]
        hits += bool(truth[c] & set(top_k))
    return hits / len(customers)

cf_scores = R.values @ S                                   # sum of similarities to what they bought
pop_scores = np.tile(popularity, (len(R), 1))              # same list for everyone
print(f"customers evaluated: {len(customers):,}")
print(f"hit rate @10, popularity:              {hit_rate(pop_scores):.1%}")
print(f"hit rate @10, item-item collaborative: {hit_rate(cf_scores):.1%}")

customers evaluated: 2,631
hit rate @10, popularity:              20.3%
hit rate @10, item-item collaborative: 30.0%


Collaborative filtering catches something the customer really bought in about **30%** of cases, against about **20%** for best-sellers: roughly half as many hits again, from nothing but purchase history.

## 6. Cold start

What about a brand-new customer with no history, or a product launched yesterday?

In [10]:
new_customer = np.zeros(len(products))
cold_scores = new_customer @ S
print("scores for a new customer are all zero:", np.allclose(cold_scores, 0))
print("fallback for new customers = the popularity list:")
for code_ in products[np.argsort(-popularity)[:5]]:
    print("  ", names[code_])

scores for a new customer are all zero: True
fallback for new customers = the popularity list:
   REGENCY CAKESTAND 3 TIER
   WHITE HANGING HEART T-LIGHT HOLDER
   PARTY BUNTING
   PACK OF 72 RETROSPOT CAKE CASES
   ASSORTED COLOUR BIRD ORNAMENT


A new customer gets all-zero scores, and a new product has no column at all. Real systems fall back to popularity, ask a few questions on sign-up, or use **content-based** similarity (product descriptions, category, price) until enough purchases arrive.

## Quiz answers

1. Recommend the most popular items the user has not already bought. It is the baseline every recommender must beat.
2. cos(a, b) = a·b / (|a| |b|): 1 means the two products were bought by exactly the same customers, 0 means no customer in common.
3. It scores each unbought product by its similarity to the products the customer already bought, using only the purchase matrix, not product details.
4. Content-based uses item attributes (description, category, price); collaborative uses only who bought what. Content-based can recommend brand-new items; collaborative finds non-obvious links but needs history.
5. A new user or item has no purchase history, so collaborative filtering has nothing to compute with. Fall back to popularity, ask for preferences, or use content-based similarity.

**Next:** [06.P Mini-project: RFM customer segments and fraud flags](../project/)